In [1]:
import numpy as np
import rasterio
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict
import plotly.graph_objects as go

# Sankey Diagram of Land Cover Transitions (2022-2025)

### INPUT FILES

In [2]:
year_files = {
    2022: "./IMAGE_DI/RES_map_2022_RGB.tif",
    2023: "./IMAGE_DI/RES_map_2023_RGB.tif",
    2024: "./IMAGE_DI/RES_map_2024_RGB.tif",
    2025: "./IMAGE_DI/RES_map_2025_RGB.tif",
}

### LAND COVER CLASSES

In [3]:
class_names = [
    "bare ground", "build up area", "crops",
    "grass&scrub", "trees", "water"
]

# RGB values used in classified rasters
class_colors_rgb = {
    0: (150, 75, 0),     # bare ground
    1: (128, 128, 128),  # built-up area
    2: (255, 255, 0),    # crops
    3: (144, 238, 144),  # grass & scrub
    4: (0, 100, 0),      # trees
    5: (0, 0, 255)       # water
}

# Hex colors used in visualization
class_colors_hex = {
    "bare ground":   "#964B00",
    "build up area": "#808080",
    "crops":         "#FFFF00",
    "grass&scrub":   "#90EE90",
    "trees":         "#006400",
    "water":         "#0000FF"
}

# Mapping RGB -> class ID lookup for raster parsing
color_to_class = {v: k for k, v in class_colors_rgb.items()}

### Load classified rasters and compute area statistics

In [4]:
results = {name: {} for name in class_names}
class_maps = {}

for year, path in year_files.items():
    # print(f"Processing raster for year: {year} ...")

    # Read RGB raster
    with rasterio.open(path) as src:
        rgb = src.read()
        rgb = np.transpose(rgb, (1, 2, 0))

    # Empty classification map (all pixels initially undefined)
    class_map = np.full(rgb.shape[:2], -1, dtype=np.int32)

    # Assign class index based on RGB matching
    for color, class_idx in color_to_class.items():
        mask = np.all(rgb == color, axis=-1)
        class_map[mask] = class_idx

    class_maps[year] = class_map

    # Compute class area share
    total_pixels = np.sum(class_map != -1)
    for i, name in enumerate(class_names):
        count = np.sum(class_map == i)
        percentage = 100 * count / total_pixels if total_pixels > 0 else 0
        results[name][year] = round(percentage, 2)

# Create a dataframe summarizing class percentages for each year
df = pd.DataFrame(results).T
df = df[[2022, 2023, 2024, 2025]]
df.index.name = "Class"

### Sort classes based on mean spatial dominance

In [5]:
mean_share = df.mean(axis=1)                  
ordered_classes = mean_share.sort_values(ascending=False).index.tolist()

### Compute year-to-year transition matrix

In [6]:
change_records = []
years_sorted = sorted(class_maps.keys())

for i in range(len(years_sorted) - 1):
    y1, y2 = years_sorted[i], years_sorted[i + 1]

    map1 = class_maps[y1]
    map2 = class_maps[y2]

    for from_idx, from_name in enumerate(class_names):
        mask = map1 == from_idx
        changed = map2[mask]
        total = np.sum(mask)

        for to_idx, to_name in enumerate(class_names):
            count = np.sum(changed == to_idx)
            perc = 100 * count / total if total > 0 else 0

            change_records.append({
                "From class": from_name,
                "To class": to_name,
                f"{y1}→{y2}": perc
            })

df_all = pd.DataFrame(change_records)

# Pivot to matrix form: (from,to) vs transition years
df_summary = df_all.pivot_table(
    index=["From class", "To class"],
    values=[col for col in df_all.columns if "→" in col],
    aggfunc="mean"
)

df_summary = df_summary[
    df_summary.index.get_level_values(0) != df_summary.index.get_level_values(1)
]

pd.set_option('display.float_format', lambda x: f'{x:.1f}%')
print("\nSummary of transitions (excluding unchanged classes):")
display(df_summary.fillna(0))


Summary of transitions (excluding unchanged classes):


2022→2023  2023→2024  2024→2025
From class    To class                                      
bare ground   build up area      11.0%      11.2%       6.3%
              crops               2.9%      16.2%       4.4%
              grass&scrub        24.7%      56.8%      46.8%
              trees               7.3%       4.8%       8.9%
              water               0.7%       0.1%       0.1%
build up area bare ground        25.9%       3.7%      13.9%
              crops               7.1%       7.8%       0.3%
              grass&scrub        21.8%      52.8%      12.2%
              trees               5.8%       6.3%       6.8%
              water               0.6%       0.2%       0.1%
crops         bare ground         5.6%       2.2%      25.9%
              build up area       9.2%       1.2%       2.3%
              grass&scrub        31.2%      73.0%      43.4%
              trees              13.3%       3.4%      24.6%
              water               0.0%       0.0%       0.0%
grass&scrub   bare ground        10.1%       3.3%       8.4%
              build up area       6.2%       1.4%       3.5%
              crops              19.4%       4.8%       1.7%
              trees              17.6%      13.8%      22.6%
              water               0.0%       0.0%       0.0%
trees         bare ground         5.2%       2.3%       4.2%
              build up area       5.4%       1.6%       5.3%
              crops              36.2%       4.3%       1.1%
              grass&scrub        17.6%      44.5%      39.5%
              water               0.1%       0.1%       0.0%
water         bare ground         6.6%       3.0%       5.9%
              build up area      13.0%       4.8%       6.4%
              crops               0.4%       0.0%       0.1%
              grass&scrub         0.1%       0.4%       0.3%
              trees               2.1%       2.9%       6.7%

### Sankey diagram

In [7]:
def build_full_sankey_colored(df_summary, ordered_classes, year_steps):
    """
    Builds a Sankey diagram showing temporal transitions between land cover classes.

    Parameters:
        df_summary      : pivot table of land cover transition percentages
        ordered_classes : class ordering sorted by spatial share (largest -> smallest)
        year_steps      : sequential years defining the temporal flow (e.g., ["2022","2023","2024","2025"])
    """
    node_labels, link_colors = [], []
    label_to_index, sources, targets, values = {}, [], [], []
    node_colors = []

    num_classes = len(ordered_classes)

    # Create labeled nodes for each year in consistent order
    for i, year in enumerate(year_steps):
        for cls in ordered_classes:
            label = f"{year}: {cls}"
            idx = i * num_classes + ordered_classes.index(cls)
            label_to_index[(year, cls)] = idx
            node_labels.append(label)
            node_colors.append(class_colors_hex.get(cls, "#D3D3D3"))

    # Build flow links between consecutive years
    for i in range(len(year_steps) - 1):
        y1, y2 = year_steps[i], year_steps[i + 1]
        col_name = f"{y1}→{y2}"

        # Skip if no transition data exists
        if col_name not in df_summary.columns:
            continue

        df_col = df_summary[[col_name]].fillna(0).reset_index()
        df_col = df_col[df_col[col_name] > 0]

        for _, row in df_col.iterrows():
            from_cls, to_cls, val = row["From class"], row["To class"], row[col_name]

            if from_cls not in ordered_classes or to_cls not in ordered_classes:
                continue

            sources.append(label_to_index[(y1, from_cls)])
            targets.append(label_to_index[(y2, to_cls)])
            values.append(val)

            # Link inherits color of the origin class
            link_colors.append(class_colors_hex.get(from_cls, "lightgray"))

    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(source=sources, target=targets, value=values, color=link_colors)
    )])

    fig.update_layout(
        font=dict(size=18, family="Arial"),
        margin=dict(l=10, r=10, t=30, b=10),
        height=700,
        width=1100,
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig.show()

In [8]:
year_steps = ["2022", "2023", "2024", "2025"]
build_full_sankey_colored(df_summary, ordered_classes, year_steps)

# Cover maps and damage index rasters

### INPUT FILES

In [9]:
damage_files = {
    2022: "./IMAGE_DI/damage_index_lt4_2022.tif",
    2023: "./IMAGE_DI/damage_index_lt4_2023.tif",
    2024: "./IMAGE_DI/damage_index_lt4_2024.tif",
    2025: "./IMAGE_DI/damage_index_lt4_2025.tif"
}

### LAND COVER CLASSES

In [10]:
class_names = ["bare ground", "build up area", "crops", "grass&scrub", "trees", "water"]

# Colors used in Sankey visualization
class_colors_hex = {
    "bare ground": "#964B00",
    "build up area": "#808080",
    "crops": "#FFFF00",
    "grass&scrub": "#90EE90",
    "trees": "#006400",
    "water": "#0000FF"
}

# Colors used in raster classification (RGB based)
class_colors_rgb = {
    (150, 75, 0): 0,
    (128, 128, 128): 1,
    (255, 255, 0): 2,
    (144, 238, 144): 3,
    (0, 100, 0): 4,
    (0, 0, 255): 5
}
color_to_class = {tuple(k): v for k, v in class_colors_rgb.items()}

### LOADING RASTERS

In [11]:
def read_class_map(path):
    """Reads a classification raster (RGB coded) and converts colors to class indices."""
    with rasterio.open(path) as src:
        rgb = np.transpose(src.read(), (1, 2, 0))  # (H,W,3)

    class_map = np.full(rgb.shape[:2], -1, dtype=np.int8)
    for color, cls in color_to_class.items():
        class_map[np.all(rgb == color, axis=-1)] = cls
    return class_map


def read_damage_map(path):
    """Reads the damage index raster."""
    with rasterio.open(path) as src:
        return src.read(1).astype(np.uint8)


# Load rasters
class_maps = {y: read_class_map(p) for y, p in year_files.items()}
damage_maps = {y: read_damage_map(p) for y, p in damage_files.items()}
years_sorted = sorted(class_maps.keys())

### COMPUTE AREA SHARE ORDER FOR CONSISTENT NODE ORDERING

In [12]:
area_stats = {cls_name: {} for cls_name in class_names}

for year in years_sorted:
    cmap = class_maps[year]
    for idx, cls_name in enumerate(class_names):
        area_stats[cls_name][year] = int(np.sum(cmap == idx))

df_area = pd.DataFrame(area_stats).T
df_area.columns = years_sorted

# Mean area per land cover class used for ordering nodes in Sankey
mean_area = df_area.mean(axis=1)
ordered_classes = mean_area.sort_values(ascending=False).index.tolist()

### COMPUTE YEAR-TO-YEAR TRANSITION MATRIX

In [13]:
change_records = []
damage_threshold = 3

for i in range(len(years_sorted) - 1):
    y1, y2 = years_sorted[i], years_sorted[i + 1]
    map1, map2 = class_maps[y1], class_maps[y2]
    dmg = damage_maps[y1]

    for from_idx, from_name in enumerate(class_names):
        mask = (map1 == from_idx) & (dmg >= damage_threshold)
        changed = map2[mask]
        total = np.sum(mask)

        for to_idx, to_name in enumerate(class_names):
            count = np.sum(changed == to_idx)
            perc = 100 * count / total if total > 0 else 0
            change_records.append({
                "From class": from_name,
                "To class": to_name,
                f"{y1}→{y2}": perc
            })

df_all = pd.DataFrame(change_records)

# Pivot to summary format
df_summary = df_all.pivot_table(
    index=["From class", "To class"],
    values=[c for c in df_all.columns if "→" in c],
    aggfunc="mean"
)

# Optionally remove diagonal
df_summary = df_summary[
    df_summary.index.get_level_values(0) != df_summary.index.get_level_values(1)
]

### SANKEY DIAGRAM GENERATOR

In [14]:
def build_full_sankey_colored(df_summary, ordered_classes, year_steps):
    """
    Creates a Sankey diagram where:
    - Nodes are ordered based on total land cover area (largest -> smallest).
    - Link colors correspond to the origin land cover class.
    """
    node_labels, node_colors = [], []
    label_to_index = {}
    sources, targets, values, link_colors = [], [], [], []

    num_classes = len(ordered_classes)

    # Create node labels
    for i, year in enumerate(year_steps):
        for cls in ordered_classes:
            node_labels.append(f"{year}: {cls}")
            node_colors.append(class_colors_hex.get(cls, "#D3D3D3"))
            label_to_index[(year, cls)] = i * num_classes + ordered_classes.index(cls)

    # Create transition links between years
    for i in range(len(year_steps) - 1):
        y1, y2 = year_steps[i], year_steps[i + 1]
        col_name = f"{y1}→{y2}"

        if col_name not in df_summary.columns:
            continue

        df_trans = df_summary[[col_name]].fillna(0).reset_index()
        df_trans = df_trans[df_trans[col_name] > 0]

        for _, row in df_trans.iterrows():
            from_cls, to_cls = row["From class"], row["To class"]
            val = row[col_name]

            if from_cls not in ordered_classes or to_cls not in ordered_classes:
                continue

            # Link nodes
            sources.append(label_to_index[(y1, from_cls)])
            targets.append(label_to_index[(y2, to_cls)])
            values.append(val)

            # Convert hex color to RGBA for semi-transparent link rendering
            hex_color = class_colors_hex.get(from_cls).lstrip('#')
            r, g, b = (int(hex_color[j:j+2], 16) for j in (0, 2, 4))
            link_colors.append(f"rgba({r},{g},{b},0.3)")

    # Plot Sankey
    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        )
    )])

    fig.update_layout(
        font=dict(size=18, family="Arial"),
        margin=dict(l=10, r=10, t=30, b=10),
        height=700,
        width=1100,
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig.show()

### GENERATE FINAL SANKEY

In [15]:
year_steps = ["2022", "2023", "2024", "2025"]

build_full_sankey_colored(df_summary, ordered_classes, year_steps)